# Pothole Patrol — YOLOv8-nano Training on Kaggle

**Datasets required (add both before running):**
- `aliabdelmenam/rdd-2022` — 47K images, already YOLO format, multi-class (we filter to potholes only)
- `anggadwisunarto/potholes-detection-yolov8` — 1.9K images, YOLO format, single-class potholes

**Strategy:** Merge both datasets, remap RDD2022 pothole class to 0, train YOLOv8-nano in one pass.

**Output:** `best.pt` + `pothole_model.tflite` in `/kaggle/working/` — download both after training.

In [ ]:
# ── 1. Install ───────────────────────────────────────────────────────────────
!pip install ultralytics==8.3.0 --quiet

In [ ]:
# ── 2. Paths & directories ───────────────────────────────────────────────────
import shutil, random, yaml
from pathlib import Path

WORK_DIR = Path('/kaggle/working')

RDD_ROOT = Path('/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT')
YV8_ROOT = Path('/kaggle/input/datasets/anggadwisunarto/potholes-detection-yolov8')

DATASET_DIR = WORK_DIR / 'dataset'
for split in ('train', 'val', 'test'):
    (DATASET_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

print('Dirs ready')
print('RDD_ROOT exists:', RDD_ROOT.exists())
print('YV8_ROOT exists:', YV8_ROOT.exists())

In [ ]:
# ── 3. Inspect RDD2022 data.yaml to find the pothole class index ─────────────
# The dataset has multiple damage classes — we need to know which index = pothole (D40)
rdd_yaml_candidates = list(RDD_ROOT.rglob('*.yaml')) + list(RDD_ROOT.rglob('*.yml'))
print('RDD yaml files:', rdd_yaml_candidates)

rdd_class_names = None
for yp in rdd_yaml_candidates:
    try:
        cfg = yaml.safe_load(yp.read_text())
        if 'names' in cfg:
            rdd_class_names = cfg['names']
            print(f'Found class names in {yp}: {rdd_class_names}')
            break
    except Exception:
        pass

if rdd_class_names is None:
    # Peek at a label file to see what class indices are used
    sample_labels = list((RDD_ROOT / 'train' / 'labels').glob('*.txt'))[:5]
    for lbl in sample_labels:
        lines = lbl.read_text().strip().splitlines()
        classes_in_file = sorted(set(int(l.split()[0]) for l in lines if l.strip()))
        print(f'{lbl.name}: classes {classes_in_file}')

In [ ]:
# ── 4. Determine pothole class index in RDD2022 ──────────────────────────────
# Standard RDD2022 class mapping:
#   0: D00 (Longitudinal Crack)
#   1: D10 (Transverse Crack)
#   2: D20 (Alligator Crack)
#   3: D40 (Pothole)  ← this is what we want
# If your yaml showed different names, update RDD_POTHOLE_CLASS_ID below.

RDD_POTHOLE_CLASS_ID = None  # will be set automatically below

if rdd_class_names:
    for idx, name in enumerate(rdd_class_names):
        if 'D40' in str(name) or 'pothole' in str(name).lower() or 'Pothole' in str(name):
            RDD_POTHOLE_CLASS_ID = idx
            print(f'Pothole class index: {idx} ("{name}")')
            break

if RDD_POTHOLE_CLASS_ID is None:
    # Fallback: standard RDD2022 ordering
    RDD_POTHOLE_CLASS_ID = 3
    print(f'Could not detect from yaml — using default class index: {RDD_POTHOLE_CLASS_ID} (D40)')

print(f'Will filter RDD2022 labels: keep class {RDD_POTHOLE_CLASS_ID} → remap to class 0')

In [ ]:
# ── 5. Copy RDD2022 pothole images + remap labels ────────────────────────────
def copy_rdd_split(rdd_split: str, dest_split: str) -> int:
    """
    For each image in RDD_ROOT/<rdd_split>/images/:
    - Read the YOLO label file
    - Keep only lines where class == RDD_POTHOLE_CLASS_ID
    - Remap that class to 0
    - Skip images with no pothole annotations
    """
    img_dir = RDD_ROOT / rdd_split / 'images'
    lbl_dir = RDD_ROOT / rdd_split / 'labels'

    if not img_dir.exists():
        print(f'  {rdd_split}: directory not found at {img_dir}')
        return 0

    copied = skipped = 0
    for img_path in sorted(img_dir.glob('*.*')):
        if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue

        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            skipped += 1
            continue

        # Filter to pothole lines only, remap class index to 0
        pothole_lines = []
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            if int(parts[0]) == RDD_POTHOLE_CLASS_ID:
                pothole_lines.append('0 ' + ' '.join(parts[1:]))

        if not pothole_lines:
            skipped += 1
            continue

        shutil.copy2(img_path, DATASET_DIR / 'images' / dest_split / f'rdd_{img_path.name}')
        (DATASET_DIR / 'labels' / dest_split / f'rdd_{img_path.stem}.txt').write_text(
            '\n'.join(pothole_lines)
        )
        copied += 1

    print(f'RDD2022/{rdd_split} → {dest_split}: {copied} pothole images, {skipped} skipped')
    return copied

copy_rdd_split('train', 'train')
copy_rdd_split('val',   'val')
copy_rdd_split('test',  'test')

In [ ]:
# ── 6. Copy potholes-detection-yolov8 dataset ────────────────────────────────
# Structure: train/images/, train/labels/, valid/images/, valid/labels/
def copy_yv8_split(src_name: str, dest_split: str) -> int:
    src_img = YV8_ROOT / src_name / 'images'
    src_lbl = YV8_ROOT / src_name / 'labels'
    if not src_img.exists():
        print(f'  {src_name}: not found')
        return 0

    copied = 0
    for img_path in sorted(src_img.glob('*.*')):
        if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        lbl_path = src_lbl / (img_path.stem + '.txt')
        if not lbl_path.exists():
            continue
        shutil.copy2(img_path, DATASET_DIR / 'images' / dest_split / f'yv8_{img_path.name}')
        shutil.copy2(lbl_path, DATASET_DIR / 'labels' / dest_split / f'yv8_{lbl_path.name}')
        copied += 1

    print(f'YOLOv8/{src_name} → {dest_split}: {copied} images')
    return copied

copy_yv8_split('train', 'train')
copy_yv8_split('valid', 'val')

In [ ]:
# ── 7. Ensure test split has enough images ───────────────────────────────────
test_count = len(list((DATASET_DIR / 'images' / 'test').glob('*.*')))
if test_count < 50:
    print(f'Test split only {test_count} images — carving 10% from val...')
    val_imgs = sorted((DATASET_DIR / 'images' / 'val').glob('*.*'))
    random.seed(42)
    sample = random.sample(val_imgs, max(50, len(val_imgs) // 10))
    for img in sample:
        lbl = DATASET_DIR / 'labels' / 'val' / (img.stem + '.txt')
        shutil.move(str(img), DATASET_DIR / 'images' / 'test' / img.name)
        if lbl.exists():
            shutil.move(str(lbl), DATASET_DIR / 'labels' / 'test' / lbl.name)
    print(f'Moved {len(sample)} images to test')

print('\n── Final dataset counts ──────────────────')
for split in ('train', 'val', 'test'):
    n = len(list((DATASET_DIR / 'images' / split).glob('*.*')))
    print(f'  {split}: {n} images')

In [ ]:
# ── 8. Write data.yaml ───────────────────────────────────────────────────────
yaml_path = WORK_DIR / 'pothole.yaml'
yaml_path.write_text(yaml.dump({
    'path':  str(DATASET_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    1,
    'names': ['pothole'],
}, default_flow_style=False))
print(yaml_path.read_text())

In [ ]:
# ── 9. Train YOLOv8-nano ─────────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=str(yaml_path),
    epochs=60,
    imgsz=320,       # matches mobile inference size
    batch=32,
    device=0,        # T4 GPU
    project=str(WORK_DIR / 'runs'),
    name='pothole',
    exist_ok=True,
    patience=15,     # early stopping
    val=True,
    # Augmentation tuned for Indian roads (varying light, wet roads, camera angles)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    flipud=0.1,
    mosaic=1.0,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
)

best_pt = WORK_DIR / 'runs' / 'pothole' / 'weights' / 'best.pt'
map50 = results.results_dict.get('metrics/mAP50(B)', 0)
print(f'\nTraining complete — mAP50: {map50:.3f}')
print(f'Best weights: {best_pt}')

In [ ]:
# ── 10. Evaluate on test split ───────────────────────────────────────────────
trained = YOLO(str(best_pt))
metrics = trained.val(data=str(yaml_path), imgsz=320, split='test')

print('\n── Test Results ────────────────────────')
print(f'  mAP50:      {metrics.box.map50:.3f}')
print(f'  mAP50-95:   {metrics.box.map:.3f}')
print(f'  Precision:  {metrics.box.mp:.3f}')
print(f'  Recall:     {metrics.box.mr:.3f}')

In [ ]:
# ── 11. Export to TFLite INT8 + copy best.pt ─────────────────────────────────
from pathlib import Path

shutil.copy2(best_pt, WORK_DIR / 'best.pt')

export_model = YOLO(str(best_pt))
tflite_path = Path(export_model.export(format='tflite', imgsz=320, int8=True))
shutil.copy2(tflite_path, WORK_DIR / 'pothole_model.tflite')

pt_kb    = (WORK_DIR / 'best.pt').stat().st_size // 1024
tflite_kb = (WORK_DIR / 'pothole_model.tflite').stat().st_size // 1024

print('\n── Output files ────────────────────────')
print(f'  best.pt              {pt_kb} KB  → upload to Railway, set ML_MODEL_PATH')
print(f'  pothole_model.tflite {tflite_kb} KB  → copy to pothole-patrol-mobile/assets/ml/')
print(f'\n  mAP50: {metrics.box.map50:.3f}')
print(f'  Precision: {metrics.box.mp:.3f}  Recall: {metrics.box.mr:.3f}')
print('\nDownload both files from the Output tab on the right.')